In [0]:
# Configuration
# Load centralized config to get all table names
UTILS_DIR = "/Workspace/Repos/ts.maira@gmail.com/techmart-catalog-pipeline/utils"
exec(open(f"{UTILS_DIR}/config.py").read())

print("✅ Config loaded")

# Bronze Contrains

## BRONZE_PRODUCTS

In [0]:
# Without product_id we cannot join with vendors or track the record
spark.table(BRONZE_PRODUCTS).display()

In [0]:
# Bronze Products constraints
# Drop existing constraints first — makes this cell idempotent
for constraint in [
    "bronze_products_id_not_null",
    "bronze_products_description_not_null",
    "bronze_products_weight_not_null",
    "bronze_products_price_not_null"
]:
    try:
        spark.sql(f"""
            ALTER TABLE {BRONZE_PRODUCTS}
            DROP CONSTRAINT IF EXISTS {constraint}
        """)
    except:
        pass

# product_id: must exist and not be empty string
# It is the primary key that connects products to vendors
spark.sql(f"""
    ALTER TABLE {BRONZE_PRODUCTS}
    ADD CONSTRAINT bronze_products_id_not_null
    CHECK (product_id IS NOT NULL AND trim(product_id) != '')
""")

# product_description_raw: must exist and not be empty
# It is the raw text the LLM will read — empty means nothing to extract
spark.sql(f"""
    ALTER TABLE {BRONZE_PRODUCTS}
    ADD CONSTRAINT bronze_products_description_not_null
    CHECK (product_description_raw IS NOT NULL
           AND trim(product_description_raw) != '')
""")

# weight_raw: must exist and not be empty
# Even if the format is messy, the raw value must be present
spark.sql(f"""
    ALTER TABLE {BRONZE_PRODUCTS}
    ADD CONSTRAINT bronze_products_weight_not_null
    CHECK (weight_raw IS NOT NULL AND trim(weight_raw) != '')
""")

# unit_price_raw: must exist and not be empty
# A product without price cannot be used in analytics
spark.sql(f"""
    ALTER TABLE {BRONZE_PRODUCTS}
    ADD CONSTRAINT bronze_products_price_not_null
    CHECK (unit_price_raw IS NOT NULL AND trim(unit_price_raw) != '')
""")

print(f"✅ Bronze Products constraints applied")

## BRONZE_VENDORS

In [0]:
# Without product_id we cannot join with vendors or track the record
spark.table(BRONZE_VENDORS).display()

In [0]:
# Bronze Vendors constraints
for constraint in [
    "bronze_vendors_id_not_null",
    "bronze_vendors_name_not_null"
]:
    try:
        spark.sql(f"""
            ALTER TABLE {BRONZE_VENDORS}
            DROP CONSTRAINT IF EXISTS {constraint}
        """)
    except:
        pass

# product_id: links vendor to product — cannot be null or empty
spark.sql(f"""
    ALTER TABLE {BRONZE_VENDORS}
    ADD CONSTRAINT bronze_vendors_id_not_null
    CHECK (product_id IS NOT NULL AND trim(product_id) != '')
""")

# vendor_name_raw: must exist — even if messy, the name must be there
spark.sql(f"""
    ALTER TABLE {BRONZE_VENDORS}
    ADD CONSTRAINT bronze_vendors_name_not_null
    CHECK (vendor_name_raw IS NOT NULL AND trim(vendor_name_raw) != '')
""")

print(f"✅ Bronze Vendors constraints applied")

# Silver Constrains

## SILVER_PRODUCTS

In [0]:
# Without product_id we cannot join with vendors or track the record
spark.table(SILVER_PRODUCTS).display()

In [0]:
# Silver Products constraints
# Silver is stricter — data has been cleaned here
# so we validate both existence AND values

for constraint in [
    "silver_products_id_not_null",
    "silver_products_description_not_null",
    "silver_products_weight_positive",
    "silver_products_price_positive"
]:
    try:
        spark.sql(f"""
            ALTER TABLE {SILVER_PRODUCTS}
            DROP CONSTRAINT IF EXISTS {constraint}
        """)
    except:
        pass

# product_id: primary key — must exist and not be empty
spark.sql(f"""
    ALTER TABLE {SILVER_PRODUCTS}
    ADD CONSTRAINT silver_products_id_not_null
    CHECK (product_id IS NOT NULL)
""")

# product_description: cleaned description — must exist and not be empty
spark.sql(f"""
    ALTER TABLE {SILVER_PRODUCTS}
    ADD CONSTRAINT silver_products_description_not_null
    CHECK (product_description IS NOT NULL
           AND trim(product_description) != '')
""")

# weight_kg: must be positive — negative or zero weight is impossible
# NULL is allowed — some products may have unparseable weight formats
spark.sql(f"""
    ALTER TABLE {SILVER_PRODUCTS}
    ADD CONSTRAINT silver_products_weight_positive
    CHECK (weight_kg IS NULL OR weight_kg > 0)
""")

# unit_price_usd: must be positive — negative price makes no business sense
# NULL is allowed — some products may have unparseable price formats
spark.sql(f"""
    ALTER TABLE {SILVER_PRODUCTS}
    ADD CONSTRAINT silver_products_price_positive
    CHECK (unit_price_usd IS NULL OR unit_price_usd > 0)
""")

print(f"✅ Silver Products constraints applied")

## SILVER_VENDORS

In [0]:
# Without product_id we cannot join with vendors or track the record
spark.table(SILVER_VENDORS).display()

In [0]:
# Silver Vendors constraints
for constraint in [
    "silver_vendors_id_not_null",
    "silver_vendors_name_not_null"
]:
    try:
        spark.sql(f"""
            ALTER TABLE {SILVER_VENDORS}
            DROP CONSTRAINT IF EXISTS {constraint}
        """)
    except:
        pass

# product_id: join key — must exist
spark.sql(f"""
    ALTER TABLE {SILVER_VENDORS}
    ADD CONSTRAINT silver_vendors_id_not_null
    CHECK (product_id IS NOT NULL)
""")

# vendor_name: normalized name — must exist and not be empty at Silver
spark.sql(f"""
    ALTER TABLE {SILVER_VENDORS}
    ADD CONSTRAINT silver_vendors_name_not_null
    CHECK (vendor_name IS NOT NULL AND trim(vendor_name) != '')
""")

print(f"✅ Silver Vendors constraints applied")

# Gold Constrains

## GOLD_SUMMARY

In [0]:
# Without product_id we cannot join with vendors or track the record
spark.table(GOLD_SUMMARY).display()

In [0]:
# Gold constraints — analytics layer must have clean aggregation data
for constraint in [
    "gold_summary_category_not_null",
    "gold_summary_products_positive"
]:
    try:
        spark.sql(f"ALTER TABLE {GOLD_SUMMARY} DROP CONSTRAINT IF EXISTS {constraint}")
    except:
        pass

# Category must always exist in the Gold aggregation
spark.sql(f"""
    ALTER TABLE {GOLD_SUMMARY}
    ADD CONSTRAINT gold_summary_category_not_null
    CHECK (category IS NOT NULL AND trim(category) != '')
""")

# Number of products must always be positive
spark.sql(f"""
    ALTER TABLE {GOLD_SUMMARY}
    ADD CONSTRAINT gold_summary_products_positive
    CHECK (num_products > 0)
""")

print(f"✅ Gold constraints applied")

# Verify all constraints were created

In [0]:
# CELL 6 — Verify all constraints were created
for label, table in [
    ("Bronze Products", BRONZE_PRODUCTS),
    ("Bronze Vendors",  BRONZE_VENDORS),
    ("Silver Products", SILVER_PRODUCTS),
    ("Silver Vendors",  SILVER_VENDORS)   # ← estava faltando
]:
    print(f"\n=== {label} ===")
    spark.sql(f"SHOW TBLPROPERTIES {table}") \
         .filter("key LIKE 'delta.constraints%'") \
         .show(truncate=False)